# 09 — Mobile nano: Luna + personal videos, at 960 px

This experiment trains **YOLO26n at 960 px**, with 1,547 train images and 168 validation images. We start again from the COCO weights: the previous model serves as a reference, not as an initialization.

The computation runs on **Hugging Face A10G**, not on the Mac. This notebook presents the configuration, the monitoring and the results; **no cell launches a new paid job**.

The 292 images of the personal test set are used neither for training nor for choosing the best checkpoint. The validation split stays separated by match groups from the Luna compilation.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "configs/mobile_nano_960.json").is_file())
CONFIG_PATH = PROJECT_ROOT / "configs/mobile_nano_960.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
JOB_PATH = PROJECT_ROOT / "models/reports" / (CONFIG["experiment_id"] + "_job.json")
JOB = json.loads(JOB_PATH.read_text()) if JOB_PATH.exists() else None
ACTUAL_CONFIG = JOB["config"] if JOB else CONFIG
RUN_ROOT = PROJECT_ROOT / "models/checkpoints/baseline_runs" / ACTUAL_CONFIG["experiment_id"]
ARTIFACT_ROOT = RUN_ROOT / "artifacts"
print("Project:", PROJECT_ROOT)
print("Results:", ARTIFACT_ROOT)

## 1. What data does it learn from?

We mix the first reviewed compilation and the three personal train clips. The old Infactory TV data is not reintroduced in this first experiment.

**37 ambiguous negatives** from train/validation (no Luna response followed by an empty label) are quarantined. Images corrected to positive stay in. No parent dataset and no human decision is modified.

The local JPEGs are hard links: no second permanent copy of the images. The validation split contains two visually identified match groups: it remains small and does not represent every personal situation.

In [ ]:
DATA = json.loads((PROJECT_ROOT / ACTUAL_CONFIG["dataset"] / "build_report.json").read_text())
lines = ["| Split | Images | With ball | Without ball |", "|---|---:|---:|---:|"]
for split, counts in DATA["splits"].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown("\n".join(lines)))
print("Quarantine:", DATA["quarantined_empty_missing_luna"])
print("Test images included:", DATA["test_images_included"])
print("Exact duplicates:", DATA["exact_image_duplicates"])
print("Train/val overlap by match:", DATA["train_val_match_overlap"])

## 2. Configuration actually launched

60 epochs maximum, stop after 12 epochs without validation progress, batch 8, 960 px images. The GPU has a hard limit of one hour. Checkpoints are saved on HF every five epochs.

`best.pt` is chosen by the validation split, **never by the test split**. The weights and images of this experiment stay private. Changing the configuration file does not change the job already launched; the cell displays its snapshot.

In [ ]:
display(Markdown("\n".join([
    f"- Model: **{ACTUAL_CONFIG['base_model']}**",
    f"- Resolution: **{ACTUAL_CONFIG['imgsz']} px**",
    f"- Maximum epochs: **{ACTUAL_CONFIG['epochs']}** ; patience: **{ACTUAL_CONFIG['patience']}**",
    f"- Batch: **{ACTUAL_CONFIG['batch']}** ; seed: **{ACTUAL_CONFIG['seed']}**",
    f"- GPU: **{ACTUAL_CONFIG['hardware']}** ; limit: **{ACTUAL_CONFIG['job_timeout']}**",
])))
if JOB:
    display(Markdown(f"[Open the Hugging Face job]({JOB['job_url']})"))
else:
    print("The job is not recorded yet. Nothing is launched from this cell.")

## 3. Monitor and retrieve

Re-running this cell only checks the status and, if the job is finished, downloads the results. It neither creates nor relaunches a job.

While it is running, the intermediate results remain accessible in the private HF repository. If the job fails or hits its limit, an intermediate checkpoint is not presented as a finished training run.

In [ ]:
from huggingface_hub import HfApi

if JOB_PATH.exists():
    JOB = json.loads(JOB_PATH.read_text())
    info = HfApi().inspect_job(job_id=JOB["job_id"])
    print("Status:", info.status.stage)
    if info.status.message:
        print(info.status.message)
    if info.status.stage == "COMPLETED":
        subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/mobile_training_hf.py"),
                        "collect", "--config", str(CONFIG_PATH)],
                       cwd=PROJECT_ROOT, check=True)
    else:
        print("The final charts will appear after collection.")
else:
    print("No job recorded for this experiment.")

## 4. Comparison at equal resolution

Three models evaluated on the **same 168 images at 960 px**: generic COCO, the previous nano trained on Luna, and the new nano trained on Luna + personal videos.

Precision, recall, F1, TP, FP, FN: **confidence ≥ 0.25 and IoU ≥ 0.50**. The mAP values integrate different confidence thresholds: they do not correspond to this single operating point.

This is a comparison of experiments (new data **and** different training), not proof isolating the effect of each new image. HF GPU times do not predict speed or battery use on an iPhone.

In [ ]:
REPORT_PATH = ARTIFACT_ROOT / "training_report.json"
REPORT = json.loads(REPORT_PATH.read_text()) if REPORT_PATH.exists() else None
if REPORT and REPORT.get("status") == "complete":
    results = REPORT["validation"]
    lines = ["| Model @960 | mAP50 | mAP50–95 | Precision | Recall | F1 | TP | FP | FN | FP on negatives |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for key, label in [("pretrained", "COCO"), ("previous_luna", "Previous Luna nano"), ("mobile", "New nano")]:
        result = results[key]
        fixed = result["fixed"]
        numbers = [result["map50"], result["map50_95"], fixed["precision"], fixed["recall"], fixed["f1"]]
        lines.append("| " + label + " | " + " | ".join(f"{x:.1%}" for x in numbers)
                     + " | " + " | ".join(str(fixed.get(k, 0)) for k in ["tp", "fp", "fn", "fp_on_negatives"]) + " |")
    display(Markdown("\n".join(lines)))
    before, after = results["previous_luna"], results["mobile"]
    display(Markdown(f"mAP50 change: **{100 * (after['map50'] - before['map50']):+.1f} points**; "
                     f"F1 change: **{100 * (after['fixed']['f1'] - before['fixed']['f1']):+.1f} points**."))
    print("Epochs completed:", REPORT["epochs_logged"])
    print("Training time:", round(REPORT["training_seconds"] / 60, 1), "minutes")
else:
    print("Final report unavailable: training or collection in progress.")

## 5. Loss and validation over the epochs

We look for a decrease in the losses and an improvement in the validation metrics. A train loss that drops while the validation durably degrades suggests overfitting; a single oscillation is not enough.

The precision/recall curves in `results.csv` come from the Ultralytics computation: their operating point may differ from the fixed 0.25 threshold in the table.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt


def show_figure(fig):
    with BytesIO() as buffer:
        fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(fig)
HISTORY_PATH = ARTIFACT_ROOT / "train/results.csv"
if HISTORY_PATH.exists():
    with HISTORY_PATH.open() as stream:
        history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]
    epochs = [row["epoch"] for row in history]
    loss_names = [name for name in ["box_loss", "cls_loss", "l1_loss", "dfl_loss"]
                  if "train/" + name in history[0] and "val/" + name in history[0]]
    fig, axes = plt.subplots(1, len(loss_names), figsize=(5 * len(loss_names), 3.6), squeeze=False)
    for axis, name in zip(axes[0], loss_names):
        for split in ["train", "val"]:
            axis.plot(epochs, [row[split + "/" + name] for row in history], label=split)
        axis.set(title=name, xlabel="Epoch")
        axis.legend()
        axis.grid(alpha=.2)
    fig.tight_layout()
    show_figure(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for axis, metrics in zip(axes, [("metrics/mAP50(B)", "metrics/mAP50-95(B)"),
                                  ("metrics/precision(B)", "metrics/recall(B)")]):
        for metric in metrics:
            if metric in history[0]:
                axis.plot(epochs, [row[metric] for row in history], label=metric)
        axis.set(xlabel="Epoch", ylim=(0, 1))
        axis.legend()
        axis.grid(alpha=.2)
    fig.tight_layout()
    show_figure(fig)
else:
    print("History available after collection.")

## 6. Confusion matrices at the same threshold

Columns = annotation; rows = prediction. Top left: correct detections (TP). Top right: false detections (FP). Bottom left: missed balls (FN).

The bottom-right cell stays empty: in object detection, not every background region is counted as a true negative. These custom matrices use exactly the thresholds of the table, unlike the standard matrices whose thresholds may differ.

In [ ]:
import base64

from IPython.display import HTML

panels = []
for key, label in [("pretrained", "COCO"), ("previous_luna", "Previous Luna nano"), ("mobile", "New nano")]:
    path = ARTIFACT_ROOT / "evaluation" / key / "fixed_confusion_matrix.png"
    if path.exists():
        encoded = base64.b64encode(path.read_bytes()).decode()
        panels.append(f'<div style="flex:1;min-width:230px"><p>{label}</p><img style="width:100%" src="data:image/png;base64,{encoded}"></div>')
if panels:
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + "".join(panels) + "</div>"))
else:
    print("Matrices available after collection.")

## 7. Precision–recall curves and threshold choice

The PR curve shows the precision/recall trade-off. The F1 curve shows the thresholds that balance the two on **the validation split**. A threshold is not chosen by optimizing on the personal test set.

In [ ]:
evaluation = ARTIFACT_ROOT / "evaluation/mobile"
for pattern in ["*PR_curve.png", "*F1_curve.png"]:
    paths = sorted(evaluation.glob(pattern))
    if paths:
        display(Image(filename=str(paths[0]), width=650))
    else:
        print("Curve not available yet:", pattern)

## 8. Final personal test — previous and new at 960 px

The model and the thresholds were frozen before looking at this test set. The 292 reviewed images include 264 positives and 28 negatives, with 279 boxes (11 multi-box images). The 88 exclusions are not evaluated.

We present strict detection (IoU ≥ 0.50, all boxes) and single-target selection (maximum confidence, IoU ≥ 0.10 or 0.20) separately. With several annotations, any annotated box can match: this test does not identify with certainty the ball in play.

**Limit: the long train set and the short test set share the September 30 match.** These images are not an additional validation set and are not used to choose new epochs or an optimal threshold.


In [ ]:
PERSONAL_ROOT = PROJECT_ROOT / "models/reports/mobile_personal_test_20261010"
PERSONAL_PATH = PERSONAL_ROOT / "report.json"
PERSONAL = json.loads(PERSONAL_PATH.read_text()) if PERSONAL_PATH.exists() else None
if PERSONAL and PERSONAL.get("metrics_status") == "complete":
    results = PERSONAL["detection"]
    lines = ["| Personal test @960 | Previous nano | New nano |",
             "|---|---:|---:|"]
    before, after = results["previous_luna"], results["mobile"]
    for label, key in [("mAP50", "map50"), ("mAP50–95", "map50_95")]:
        lines.append(f"| {label} | {before[key]:.1%} | {after[key]:.1%} |")
    for label, key in [("Precision", "precision"), ("Recall", "recall"), ("F1", "f1")]:
        lines.append(f"| {label} | {before['fixed'][key]:.1%} | {after['fixed'][key]:.1%} |")
    for label, key in [("Correct detections", "tp"), ("False detections", "fp"),
                       ("Missed balls", "fn"), ("FP on negatives", "fp_on_negatives")]:
        lines.append(f"| {label} | {before['fixed'].get(key, 0)} | {after['fixed'].get(key, 0)} |")
    display(Markdown("\n".join(lines)))
    lines = ["| Framing: single target, conf ≥ 0.25 | Previous nano | New nano |",
             "|---|---:|---:|"]
    for overlap in ["0.1", "0.2"]:
        before = PERSONAL["targeting"][overlap]["previous_luna"]["metrics"]
        after = PERSONAL["targeting"][overlap]["mobile"]["metrics"]
        lines.append(f"| Correct decisions, IoU ≥ {overlap} | {before['correct_decision_rate']:.1%} | {after['correct_decision_rate']:.1%} |")
    display(Markdown("\n".join(lines)))
    for key in ["previous_luna", "mobile"]:
        path = PERSONAL_ROOT / key / "fixed_confusion_matrix.png"
        if path.exists():
            display(Image(filename=str(path), width=450))
else:
    print("Personal evaluation in progress or not yet run.")

## 9. Review the short video

The new nano analyzes **all the frames of the clip**, not only the annotated ones. A single box is displayed: the most confident one above 0.25. The green frame is slightly enlarged to be visible; the evaluated coordinates remain the original coordinates.

The confidence is in a band above the image, not in the small box. There is neither a tracker nor temporal smoothing: target jumps and disappearances remain possible. Frames excluded from the benchmark appear in the full video without being added to the metrics.

Inference is sequential on CPU, with no new paid HF computation. The video stays a separate file so the notebook does not grow.

In [ ]:
from IPython.display import Video

if PERSONAL_PATH.exists():
    PERSONAL = json.loads(PERSONAL_PATH.read_text())
if PERSONAL and PERSONAL.get("status") == "complete":
    VIDEO_PATH = Path(PERSONAL["video"]["path"])
    print("Video:", VIDEO_PATH)
    print("Duration:", round(PERSONAL["video"]["duration_seconds"], 1), "s")
    display(Video(filename=str(VIDEO_PATH), embed=False, width=960))
else:
    print("The video will be available once inference has finished.")

## 10. Controlled experiment: adding the old TV images

We add **3,048 v002 train images** to the **1,547 recent images**, without changing the **168 validation images**. No old val/test data is added to train. YOLO26n stays at 960 px and starts again from the same COCO weights; this is not a continuation of the first model.

Same recipe, batch and maximum number of epochs; but more updates per epoch and more negatives. This test measures the value of this specific addition, not only the quality of the annotations. The personal benchmark has already been inspected and is not an independent per-match test.

The cell below only reads local files: no GPU launch and no new inference. After the job finishes, retrieve its results with `python scripts/mobile_training_hf.py collect --config configs/mobile_nano_tv_960.json`, then re-run this cell.

In [ ]:
TV_CONFIG = json.loads((PROJECT_ROOT / 'configs/mobile_nano_tv_960.json').read_text())
TV_DATA = json.loads((PROJECT_ROOT / TV_CONFIG['dataset'] / 'build_report.json').read_text())
assert TV_DATA['validation_identical_to_baseline'] and TV_DATA['test_images_included'] == 0
display(Markdown('| Data | Recent only | Recent + TV |\n|---|---:|---:|\n| Train | 1,547 | 4,595 |\n| Identical validation | 168 | 168 |'))
TV_JOB_PATH = PROJECT_ROOT / 'models/reports' / (TV_CONFIG['experiment_id'] + '_job.json')
if TV_JOB_PATH.exists():
    TV_JOB = json.loads(TV_JOB_PATH.read_text())
    print('Last recorded status (not live tracking):', TV_JOB['status'])
    if TV_JOB.get('job_url'):
        display(Markdown('[Open the HF job](' + TV_JOB['job_url'] + ')'))
TV_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / TV_CONFIG['experiment_id'] / 'artifacts'
TV_REPORT_PATH = TV_ARTIFACTS / 'training_report.json'
if TV_REPORT_PATH.exists():
    TV_REPORT = json.loads(TV_REPORT_PATH.read_text())
    if TV_REPORT['status'] == 'complete':
        before, after = TV_REPORT['validation']['modern_only'], TV_REPORT['validation']['mobile']
        lines = ['| Same validation @960 | Recent only | Recent + TV | Change |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            lines.append(f"| {title} | {before[key]:.1%} | {after[key]:.1%} | {(after[key]-before[key])*100:+.1f} pts |")
        for title, key in [('Precision', 'precision'), ('Recall', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Correct detections', 'tp'), ('False detections', 'fp'), ('Missed balls', 'fn'), ('FP on negatives', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        for name in ['modern_only', 'mobile']:
            image_path = TV_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if image_path.exists():
                display(Image(filename=str(image_path), width=450))
        learning_plot = TV_ARTIFACTS / 'train/results.png'
        if learning_plot.exists():
            display(Image(filename=str(learning_plot), width=1000))
    else:
        print('Intermediate results: no final conclusion yet.')
else:
    print('New training not collected: no final metric to compare yet.')

## 11. Recent only: truly keeping the sports ball class

Same **1,547 train / 168 validation**, no TV, same settings at 960 px. We keep the characteristics of the COCO classifier and its `sports ball` output weights: **class 32 → 0** in both branches, at the three scales. This is not the COCO person class.

A simple identical name only keeps the biases here, because the single-class constructor reduces the classifier width from 80 to 64. We keep the width of 80: **+1.19% parameters**, so this test covers the full transfer, not just the renaming. Weights and raw predictions are verified before training; they may evolve afterwards.

The cell only reads the local reports: no paid computation or inference. After the job finishes: `python scripts/mobile_training_hf.py collect --config configs/mobile_nano_sports_ball_960.json`, then re-run this cell. The class name must not be mistaken for a guarantee of performance on close-ups.

In [ ]:
SPORT_CONFIG = json.loads((PROJECT_ROOT / 'configs/mobile_nano_sports_ball_960.json').read_text())
SPORT_JOB_PATH = PROJECT_ROOT / 'models/reports' / (SPORT_CONFIG['experiment_id'] + '_job.json')
if SPORT_JOB_PATH.exists():
    SPORT_JOB = json.loads(SPORT_JOB_PATH.read_text())
    print('Last recorded status (not live):', SPORT_JOB['status'])
    if SPORT_JOB.get('job_url'):
        display(Markdown('[HF sports ball job](' + SPORT_JOB['job_url'] + ')'))
    SPORT_CHECK = SPORT_JOB.get('transfer_preflight')
    if SPORT_CHECK:
        print('Class', SPORT_CHECK['source_class_id'], '→', SPORT_CHECK['target_class_id'])
        print('Weights kept:', SPORT_CHECK['all_weight_checks_passed'], '; raw predictions:', SPORT_CHECK['dense_prediction_check']['passed'])
        print('Parameters:', SPORT_CHECK['target_parameters'])
SPORT_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / SPORT_CONFIG['experiment_id'] / 'artifacts'
SPORT_REPORT_PATH = SPORT_ARTIFACTS / 'training_report.json'
if SPORT_REPORT_PATH.exists():
    SPORT_REPORT = json.loads(SPORT_REPORT_PATH.read_text())
    if SPORT_REPORT['status'] == 'complete':
        before, after = SPORT_REPORT['validation']['modern_only'], SPORT_REPORT['validation']['mobile']
        lines = ['| Same validation @960 | Previous recent | Sports ball transfer | Change |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Precision', 'precision'), ('Recall', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Correct detections', 'tp'), ('False detections', 'fp'), ('Missed balls', 'fn'), ('FP on negatives', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        for name in ['modern_only', 'mobile']:
            path = SPORT_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if path.exists():
                display(Image(filename=str(path), width=450))
        path = SPORT_ARTIFACTS / 'train/results.png'
        if path.exists():
            display(Image(filename=str(path), width=1000))
    else:
        print('Intermediate report: no final result yet.')
else:
    print('Sports ball results not collected yet.')

## 12. Enriched data: reviewed shorts + personal close-ups

**2,595 train images**: 1,547 modern ones already used + 603 verified shorts + 445 close-ups. All 456 close-ups were reviewed; only the 11 exclusions are removed, with no confidence filter. Human corrections take priority, unchanged proposals accepted upon confirmation, no draft present.

Same validation set of 168 images, no test data added to train, no Infactory TV image. All the close-ups stay in train as a single session; another session will be needed to measure their generalization. The matches of the shorts are not all identified: exact duplicates are checked, but this does not rule out an overlap of cropped scenes.

YOLO26n at **960 px**, COCO with verified full transfer of sports ball (32 → 0). We start again from COCO; the interrupted best sports ball v001 weights serve only as a before/after reference. **30 epochs maximum, patience 8**, batch 8. Private HF A10G, hard limit 90 minutes, remote save every two epochs. The data volume and the optimization schedule both change.

The next cell checks the status and retrieves the results only if the job is finished: **it never launches a paid training run**. The losses, matrices and metrics will then appear here.

**Retrieved result:** 25 epochs, early stop after eight epochs without improvement; best weights at epoch 17. About 19 min 44 s of training, excluding preparation and evaluations. At the 0.25 threshold, the enrichment finds seven more balls, but adds 25 false detections. F1: 74.1% → 70.5%; mAP50: 76.3% → 73.7%. This is not an overall improvement on this validation set, and it does not settle the performance on new close-ups. Both models are kept.

The train losses keep decreasing while the validation plateaus and fluctuates: more epochs are not justified by this experiment. This alone is not proof of overfitting. **Warning:** the precision/recall values in `train/results.png` follow the operating point chosen by Ultralytics, not necessarily the fixed 0.25 threshold of the table below.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from huggingface_hub import HfApi
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/mobile_nano_enriched_sports_ball_960.json').exists())
ENRICHED_CONFIG_PATH = PROJECT_ROOT / 'configs/mobile_nano_enriched_sports_ball_960.json'
ENRICHED_CONFIG = json.loads(ENRICHED_CONFIG_PATH.read_text())
ENRICHED_DATA = json.loads((PROJECT_ROOT / ENRICHED_CONFIG['dataset'] / 'build_report.json').read_text())
assert ENRICHED_DATA['validation_identical_to_baseline'] and ENRICHED_DATA['test_images_included'] == 0 and ENRICHED_DATA['tv_images_included'] == 0
lines = ['| Split | Images | With ball | Without ball |', '|---|---:|---:|---:|']
for split, counts in ENRICHED_DATA['splits'].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown('\n'.join(lines)))
print('Sources:', ENRICHED_DATA['sources'])
ENRICHED_JOB_PATH = PROJECT_ROOT / 'models/reports' / (ENRICHED_CONFIG['experiment_id'] + '_job.json')
if ENRICHED_JOB_PATH.exists():
    ENRICHED_JOB = json.loads(ENRICHED_JOB_PATH.read_text())
    if ENRICHED_JOB.get('job_id'):
        info = HfApi().inspect_job(job_id=ENRICHED_JOB['job_id'])
        print('Current status:', info.status.stage)
        display(Markdown('[Open the HF job](' + ENRICHED_JOB['job_url'] + ')'))
        if info.status.stage == 'COMPLETED':
            subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/mobile_training_hf.py'), 'collect', '--config', str(ENRICHED_CONFIG_PATH)], cwd=PROJECT_ROOT, check=True)
        elif info.status.message:
            print(info.status.message)
    else:
        print('No job started:', ENRICHED_JOB['status'], ENRICHED_JOB.get('message', ''))
else:
    print('No job recorded; this cell does not launch it.')
ENRICHED_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / ENRICHED_CONFIG['experiment_id'] / 'artifacts'

In [ ]:
ENRICHED_REPORT_PATH = ENRICHED_ARTIFACTS / 'training_report.json'
if ENRICHED_REPORT_PATH.exists():
    report = json.loads(ENRICHED_REPORT_PATH.read_text())
    if report['status'] == 'complete':
        before, after = report['validation']['sports_ball_v001'], report['validation']['mobile']
        lines = ['| Same validation @960 | Sports ball v001 | Enriched v002 | Change |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Precision', 'precision'), ('Recall', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Correct detections', 'tp'), ('False detections', 'fp'), ('Missed balls', 'fn'), ('FP on negatives', 'fp_on_negatives')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        print('Confidence ≥ 0.25; matching IoU ≥ 0.50. mAP: all thresholds.')
        print('Epochs completed:', report['epochs_logged'])
        for name in ['sports_ball_v001', 'mobile']:
            path = ENRICHED_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if path.exists():
                display(Image(filename=str(path), width=450))
        for relative in ['train/results.png', 'evaluation/mobile/BoxPR_curve.png', 'evaluation/mobile/BoxF1_curve.png']:
            path = ENRICHED_ARTIFACTS / relative
            if path.exists():
                display(Image(filename=str(path), width=1000))
        print('Weights:', ENRICHED_ARTIFACTS / 'best.pt')
    else:
        print('Intermediate report: do not conclude before the end.')
else:
    print('Results not collected yet. Run the status cell after the job finishes.')

## 13. Fixing nearby false positives: home negatives

The nine clips IMG_7016–IMG_7024 are about 12 seconds long. They are confirmed by the owner to contain no ball at all; visual inspection of the start/middle excerpts. We extract **one frame out of six: 63 images**, without resizing, with explicitly empty YOLO labels. No Luna call and no extra negative class.

**2,658 train**, of which 2,274 positives and 384 negatives; **168 identical val**. All close-ups and shorts stay included. No TV, no test data added to train. The nine new videos are all in train, as a single home session. The before/after diagnostic on these 63 frames therefore measures learning, **not generalization**: other objects/takes will be needed for that.

We continue from the **enriched v002 best.pt**, to keep its gains on close-up balls. This is neither a return to COCO nor a resumption of the old optimizer. The weights and the two branches of the 80-channel classifier are kept and verified before optimization. 960 px, **15 epochs maximum**, patience 5, AdamW **lr0=0.0002** (five times smaller), batch 8. Private HF A10G, hard limit 45 minutes; remote checkpoint every two epochs. The choice of the best checkpoint is still based on the football validation set.

These cells never launch a paid job: they check its status, retrieve the results if it is finished and present the measurements. We watch both the false positives and the recall of real balls.

**Retrieved result:** 14 epochs, best weights at epoch 9, about 11 min 26 s of training. On the 63 **train** negatives, 91 false boxes on 51 images before → zero after. This demonstrates learning of these scenes, not generalization. On the unchanged football validation set: F1 70.5% → 75.3%, precision 71.5% → 85.7%, recall 69.4% → 67.2%. It removes 22 false positives but misses three more balls. The added negatives and the fine-tuning change together; their respective contributions are not isolated. Check the close-ups and other objects on new takes.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from huggingface_hub import HfApi
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/mobile_nano_hard_negatives_960.json').exists())
NEGATIVE_CONFIG_PATH = PROJECT_ROOT / 'configs/mobile_nano_hard_negatives_960.json'
NEGATIVE_CONFIG = json.loads(NEGATIVE_CONFIG_PATH.read_text())
NEGATIVE_DATA = json.loads((PROJECT_ROOT / NEGATIVE_CONFIG['dataset'] / 'build_report.json').read_text())
assert NEGATIVE_DATA['validation_identical_to_baseline'] and NEGATIVE_DATA['test_images_included'] == 0 and NEGATIVE_DATA['tv_images_included'] == 0
lines = ['| Split | Images | With ball | Without ball |', '|---|---:|---:|---:|']
for split, counts in NEGATIVE_DATA['splits'].items():
    lines.append(f"| {split} | {counts['images']} | {counts['positive']} | {counts['negative']} |")
display(Markdown('\n'.join(lines)))
print('Initialization: best enriched v002 weights; lr0 =', NEGATIVE_CONFIG['lr0'])
NEGATIVE_JOB_PATH = PROJECT_ROOT / 'models/reports' / (NEGATIVE_CONFIG['experiment_id'] + '_job.json')
if NEGATIVE_JOB_PATH.exists():
    job = json.loads(NEGATIVE_JOB_PATH.read_text())
    if job.get('job_id'):
        info = HfApi().inspect_job(job_id=job['job_id'])
        print('Current status:', info.status.stage)
        display(Markdown('[Open the HF job](' + job['job_url'] + ')'))
        if info.status.stage == 'COMPLETED':
            subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/mobile_training_hf.py'), 'collect', '--config', str(NEGATIVE_CONFIG_PATH)], cwd=PROJECT_ROOT, check=True)
        elif info.status.message:
            print(info.status.message)
    else:
        print('No job started:', job['status'], job.get('message', ''))
else:
    print('No job recorded; this cell does not launch it.')
NEGATIVE_ARTIFACTS = PROJECT_ROOT / 'models/checkpoints/baseline_runs' / NEGATIVE_CONFIG['experiment_id'] / 'artifacts'

In [ ]:
path = NEGATIVE_ARTIFACTS / 'training_report.json'
if path.exists():
    report = json.loads(path.read_text())
    if report['status'] == 'complete':
        before, after = report['validation']['enriched_v002'], report['validation']['mobile']
        lines = ['| Identical football validation @960 | Enriched v002 | Negatives v003 | Change |', '|---|---:|---:|---:|']
        for title, key in [('mAP50', 'map50'), ('mAP50–95', 'map50_95')]:
            b, a = before[key], after[key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Precision', 'precision'), ('Recall', 'recall'), ('F1', 'f1')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b:.1%} | {a:.1%} | {(a-b)*100:+.1f} pts |')
        for title, key in [('Correct detections', 'tp'), ('False detections', 'fp'), ('Missed balls', 'fn')]:
            b, a = before['fixed'][key], after['fixed'][key]
            lines.append(f'| {title} | {b} | {a} | {a-b:+d} |')
        display(Markdown('\n'.join(lines)))
        print('Confidence ≥ 0.25; matching IoU ≥ 0.50. mAP: all thresholds.')
        diagnostic = report.get('negative_training_diagnostic', {})
        if diagnostic:
            b, a = diagnostic['enriched_v002'], diagnostic['mobile']
            assert b['images'] == a['images'] == 63 and not a['independent_generalization_test']
            display(Markdown(f"### Diagnostic on the 63 TRAIN negatives — not an independent test\n\n| Metric at conf ≥ 0.25 | Before | After |\n|---|---:|---:|\n| False boxes | {b['fp']} | {a['fp']} |\n| Images with a false box | {b['images_with_fp']} | {a['images_with_fp']} |\n| Share of images with FP | {b['fraction_images_with_fp']:.1%} | {a['fraction_images_with_fp']:.1%} |"))
        print('Epochs completed:', report['epochs_logged'])
        for name in ['enriched_v002', 'mobile']:
            image = NEGATIVE_ARTIFACTS / 'evaluation' / name / 'fixed_confusion_matrix.png'
            if image.exists():
                display(Image(filename=str(image), width=450))
        learning = NEGATIVE_ARTIFACTS / 'train/results.png'
        if learning.exists():
            display(Image(filename=str(learning), width=1000))
        print('New weights:', NEGATIVE_ARTIFACTS / 'best.pt')
    else:
        print('Intermediate report: wait for the end before concluding.')
else:
    print('Results not collected yet. Re-run the status cell once the job is finished.')